
# people_segment.csv → persona_attributes_weighted.jsonl (auto-detect demographics)

**실제 `people_segment.csv`의 컬럼명에 맞춰 자동으로 인구통계 컬럼을 감지**하고,  
`*_scaled` 성향 컬럼과 함께 **가중치 합=1.0**이 되도록 정규화한 JSONL을 생성합니다.

- 입력: `people_segment.csv`
- 출력: `persona_attributes_weighted.jsonl`, `persona_attributes_weighted_preview.json`
- 인구통계/성향 비중: `DEMO_SHARE`, `BEHAV_SHARE` (합=1 권장)
- 필요시 `DEMO_COLS_OVERRIDE`로 인구통계 컬럼을 직접 지정 가능


In [ ]:

# =============================
# 0) CONFIG — 조정 가능한 설정
# =============================
from pathlib import Path

# 입력/출력 경로
INPUT_CSV = Path("people_segment.csv")
OUT_JSONL = Path("persona_attributes_weighted.jsonl")
OUT_PREVIEW = Path("persona_attributes_weighted_preview.json")

# 인구통계 vs 성향 전체 비중 (합=1.0 권장)
DEMO_SHARE = 0.30
BEHAV_SHARE = 0.70

# (선택) 인구통계 컬럼 직접 지정 — 지정하면 자동탐지 대신 이 목록을 사용합니다.
# 예: ["성별","연령","지역","가족구성","학력","직업","혼인"]
DEMO_COLS_OVERRIDE = []

# 인구통계 내부 항목별 가중 (없으면 균등 분배)
DEMO_FIELD_WEIGHTS = {}

# 성향 내부 항목 multiplier (정규화 전 강조/약화) — 키는 *_scaled 컬럼명
BEHAV_FIELD_MULTIPLIERS = {}

print("CONFIG loaded. Adjust DEMO_SHARE / BEHAV_SHARE or override columns if needed.")


In [ ]:

# =============================
# 1) Load CSV & clean columns
# =============================
import pandas as pd, re

def read_csv_fallback(path: Path):
    try:
        return pd.read_csv(path)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="cp949")

df = read_csv_fallback(INPUT_CSV)

def clean_colname(c: str) -> str:
    # 'brand_loyalty' -> brand_loyalty
    return re.sub(r"'([^']+)'", r"\1", c)

df.rename(columns={c: clean_colname(c) for c in df.columns}, inplace=True)

print("Rows:", len(df))
print("Columns:", len(df.columns))
df.head(3)


In [ ]:

# =============================
# 2) Auto-detect demographics & behavioral columns
# =============================
import numpy as np

# behavior: *_scaled 컬럼 자동 선택
behav_cols = [c for c in df.columns if c.endswith("_scaled")]

# demographics 자동 감지 규칙:
# 1) 문자열(object) 타입 & _scaled로 끝나지 않는 컬럼
# 2) id/cluster/설명성 칼럼은 제외
# 3) 너무 고유값이 많은(=개인 텍스트 같은) 칼럼 제외: 고유값 비율 < 0.5
blacklist = {"id","cluster","segment_id","label","desc","Unnamed: 0"}
cat_candidates = []
for c in df.columns:
    if c in blacklist or c.endswith("_scaled"):
        continue
    if df[c].dtype == "object":
        nun = df[c].nunique(dropna=False)
        ratio = nun / max(1, len(df))
        if ratio <= 0.5:  # 절반 이상이 유니크면 카테고리로 보기 어려움
            cat_candidates.append(c)

# 숫자형인데 사실상 범주(유니크 수가 적은)인 컬럼도 보조로 포함
for c in df.columns:
    if c in blacklist or c.endswith("_scaled"):
        continue
    if np.issubdtype(df[c].dtype, np.number):
        nun = df[c].nunique(dropna=False)
        if 2 <= nun <= 20:
            cat_candidates.append(c)

# override가 있으면 override 우선
if isinstance(DEMO_COLS_OVERRIDE, (list, tuple)) and len(DEMO_COLS_OVERRIDE) > 0:
    demo_cols = [c for c in DEMO_COLS_OVERRIDE if c in df.columns]
else:
    # 상위 7개까지만 사용 (너무 많으면 프롬프트 길어짐)
    # 우선순위: 문자열 후보 먼저, 그 다음 숫자형 소범주
    # 중복 제거 순서를 유지
    seen = set()
    demo_cols = []
    for c in cat_candidates:
        if c not in seen:
            seen.add(c); demo_cols.append(c)
        if len(demo_cols) >= 7: break

if not behav_cols:
    raise RuntimeError("No behavioral (_scaled) columns detected. Please check the CSV headers.")

print("Detected Demographics columns:", demo_cols)
print("Detected Behavioral columns:", behav_cols)

# 참고: id 컬럼 존재 여부
id_like = [c for c in ["id","persona_key","person_id"] if c in df.columns]
print("ID-like columns:", id_like)


In [ ]:

# =============================
# 3) Helpers
# =============================
from typing import Dict, Any

def normalize_weights(raw: Dict[str, float], target_sum: float) -> Dict[str, float]:
    total = sum(v for v in raw.values() if pd.notna(v))
    if total <= 0:
        n = len(raw)
        return {k: (target_sum / n if n else 0.0) for k in raw}
    return {k: (v / total) * target_sum for k, v in raw.items()}

def build_persona_attributes(row: pd.Series,
                             demo_cols, behav_cols,
                             demo_share: float, behav_share: float,
                             demo_field_weights: Dict[str, float],
                             behav_field_multipliers: Dict[str, float]) -> Dict[str, Any]:
    # Demographics — 내부 가중 지정 없으면 균등(=1.0)로 시작 후 group 정규화
    demo_raw = {c: demo_field_weights.get(c, 1.0) for c in demo_cols}
    demo_weights = normalize_weights(demo_raw, demo_share)
    demo_attrs = {
        c: {
            "value": (None if pd.isna(row.get(c)) else row.get(c)),
            "weight": float(demo_weights.get(c, 0.0))
        }
        for c in demo_cols
    }

    # Behavioral — scaled 값 × multiplier → group 정규화
    behav_raw = {}
    for c in behav_cols:
        val = row.get(c)
        if pd.isna(val):
            val = 0.0
        mult = behav_field_multipliers.get(c, 1.0)
        behav_raw[c] = float(val) * float(mult)

    behav_weights = normalize_weights(behav_raw, behav_share)
    behav_attrs = {
        c: {
            "value": float(row.get(c, 0.0)),
            "weight": float(behav_weights.get(c, 0.0))
        }
        for c in behav_cols
    }

    # Merge & final re-scale to sum ~ 1.0 exactly
    attributes = {**demo_attrs, **behav_attrs}
    total_weight = sum(v["weight"] for v in attributes.values())
    if total_weight > 0:
        for k in attributes:
            attributes[k]["weight"] = attributes[k]["weight"] / total_weight
    return attributes


In [ ]:

# =============================
# 4) Build weighted attributes & save
# =============================
import json, random

total_share = DEMO_SHARE + BEHAV_SHARE
if abs(total_share - 1.0) > 1e-8:
    print(f"[WARN] DEMO_SHARE + BEHAV_SHARE = {total_share:.4f}. 자동 정규화합니다.")
    DEMO_SHARE = DEMO_SHARE / total_share
    BEHAV_SHARE = BEHAV_SHARE / total_share
    print(f" -> DEMO_SHARE={DEMO_SHARE:.4f}, BEHAV_SHARE={BEHAV_SHARE:.4f}")

records = []
id_col = "id" if "id" in df.columns else None

for idx, row in df.iterrows():
    key = row.get(id_col) if id_col else f"row_{idx}"
    persona = {
        "persona_key": key,
        "attributes": build_persona_attributes(row,
                                              demo_cols, behav_cols,
                                              DEMO_SHARE, BEHAV_SHARE,
                                              DEMO_FIELD_WEIGHTS,
                                              BEHAV_FIELD_MULTIPLIERS)
    }
    records.append(persona)

with open(OUT_JSONL, "w", encoding="utf-8") as f:
    for rec in records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"Saved JSONL -> {OUT_JSONL}")
print("Total personas:", len(records))

# sanity check few
for s in records[:3]:
    sw = sum(v["weight"] for v in s["attributes"].values())
    print("sum(weights) =", round(sw, 6))


In [ ]:

# =============================
# 5) Preview
# =============================
import json
preview = records[:3]
with open(OUT_PREVIEW, "w", encoding="utf-8") as f:
    f.write(json.dumps(preview, ensure_ascii=False, indent=2))
print(f"Saved preview -> {OUT_PREVIEW}")
preview
